# National NBI Download — Notebook #1

Downloads National Bridge Inventory (NBI) delimited ASCII files from FHWA for all 50 US states + DC, years 1992–2025, and saves one CSV per state to `raw/`.

**Source:** https://www.fhwa.dot.gov/bridge/nbi/ascii.cfm  
**URL pattern:** `https://www.fhwa.dot.gov/bridge/nbi/{year}/delimited/{STATE}{yy}.txt`

National equivalent of the Massachusetts-only download in `../bridge_project.ipynb`. The per-state CSVs (~10–80 MB each, ~15–20 GB total) are the primary output; downstream phases process them one state at a time, so they are never combined into a single file.

The download is resumable: states with a complete `raw/{STATE}_bridges.csv` are skipped, and partially downloaded states fetch only their missing years.

In [1]:
import io
import os
import csv
import time
import warnings
import requests
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# Backstop: pandas 3 emits Pandas4Warning deprecation notices; over a
# 1,734-file download loop they can bloat the cell output enormously.
from pandas.errors import Pandas4Warning
warnings.filterwarnings("ignore", category=Pandas4Warning)

In [2]:
# ── Configuration ────────────────────────────────────────────────────────────
START_YEAR  = 1992
END_YEAR    = 2025
MAX_WORKERS = 8

RAW_DIR  = Path("raw")
LOG_FILE = Path("download_log.txt")

BASE_URL = "https://www.fhwa.dot.gov/bridge/nbi/{year}/delimited/{state}{yy}.txt"
HEADERS  = {"User-Agent": "Mozilla/5.0 (research data download script)"}

RAW_DIR.mkdir(exist_ok=True)

In [3]:
# ── State lookup: (postal abbreviation, FIPS code) ───────────────────────────
# FIPS codes match NBI STATE_CODE_001 field (zero-padded to 2 digits)
STATES = [
    ("AL", "01"), ("AK", "02"), ("AZ", "04"), ("AR", "05"), ("CA", "06"),
    ("CO", "08"), ("CT", "09"), ("DE", "10"), ("DC", "11"), ("FL", "12"),
    ("GA", "13"), ("HI", "15"), ("ID", "16"), ("IL", "17"), ("IN", "18"),
    ("IA", "19"), ("KS", "20"), ("KY", "21"), ("LA", "22"), ("ME", "23"),
    ("MD", "24"), ("MA", "25"), ("MI", "26"), ("MN", "27"), ("MS", "28"),
    ("MO", "29"), ("MT", "30"), ("NE", "31"), ("NV", "32"), ("NH", "33"),
    ("NJ", "34"), ("NM", "35"), ("NY", "36"), ("NC", "37"), ("ND", "38"),
    ("OH", "39"), ("OK", "40"), ("OR", "41"), ("PA", "42"), ("RI", "44"),
    ("SC", "45"), ("SD", "46"), ("TN", "47"), ("TX", "48"), ("UT", "49"),
    ("VT", "50"), ("VA", "51"), ("WA", "53"), ("WV", "54"), ("WI", "55"),
    ("WY", "56"),
]
print(f"States to download: {len(STATES)}")
print(f"Years: {START_YEAR}–{END_YEAR} ({END_YEAR - START_YEAR + 1} years)")
print(f"Total downloads: {len(STATES) * (END_YEAR - START_YEAR + 1):,}")

States to download: 51
Years: 1992–2025 (34 years)
Total downloads: 1,734


In [4]:
# ── Download function (one state-year) ───────────────────────────────────────
def download_state_year(state_abbr: str, fips: str, year: int) -> tuple[object, int, str | None]:
    """
    Returns (df_or_None, row_count, error_message_or_None).
    Does not raise; all failures are returned as error strings.

    Parsing note: NBI delimited files use a single quote (') as the text
    qualifier, but text fields also contain literal apostrophes (e.g. "O'BRIEN",
    or a trailing "SR76''"). Passing quotechar="'" makes pandas treat those
    apostrophes as quote boundaries and desync the parser, raising
    "Expected 134 fields, saw 137". Verified across both the old (134-col) and
    new (123-col) schema years that NO field actually contains a comma, so we
    parse with QUOTE_NONE (split only on commas, treat quotes as literal text)
    and strip the decorative single quotes afterward.

    Rare exception: a few individual rows carry a genuine extra comma (e.g.
    PA 2024 line 9304 is one truly malformed record). QUOTE_NONE cannot merge a
    real extra delimiter, so rather than discard the whole file we retry with
    on_bad_lines="skip", dropping only the malformed row(s) and keeping every
    clean record. Verified to trigger on exactly one row in the whole 1,734-file
    corpus, so it never silently loses bulk data.
    """
    yy  = str(year)[-2:].zfill(2)
    url = BASE_URL.format(year=year, state=state_abbr, yy=yy)
    try:
        resp = requests.get(url, headers=HEADERS, timeout=90)
    except requests.RequestException as e:
        return None, 0, f"{year} network error: {e}"

    if resp.status_code == 404:
        return None, 0, f"{year} 404 (file not published)"
    if resp.status_code != 200:
        return None, 0, f"{year} HTTP {resp.status_code}"

    text = resp.content.decode("latin-1", errors="replace")
    try:
        df = pd.read_csv(io.StringIO(text), quoting=csv.QUOTE_NONE, low_memory=False)
    except Exception:
        # A genuine extra delimiter in one or more rows. Skip only those rows
        # (keeping every clean record) and report how many were dropped.
        n_data = text.rstrip("\n").count("\n")  # data rows (excludes header)
        try:
            df = pd.read_csv(io.StringIO(text), quoting=csv.QUOTE_NONE,
                             low_memory=False, on_bad_lines="skip")
        except Exception as e:
            return None, 0, f"{year} parse error: {e}"
        n_skipped = max(0, n_data - len(df))
        print(f"  {state_abbr} {year}: kept {len(df):,} rows, skipped {n_skipped} malformed row(s).")
    # "str" listed explicitly: pandas 3 gives text columns the new str dtype,
    # and selecting only "object" emits a Pandas4Warning per file.
    for col in df.select_dtypes(include=["object", "str"]).columns:
        df[col] = df[col].str.strip("'")

    df["STATE_FIPS"]  = fips        # explicit FIPS tag
    df["SURVEY_YEAR"] = year
    return df, len(df), None

In [5]:
# ── Download all years for one state, save to raw/{STATE}_bridges.csv ────────
def download_state(state_abbr: str, fips: str) -> dict:
    out_path = RAW_DIR / f"{state_abbr}_bridges.csv"
    if out_path.exists():
        existing = pd.read_csv(out_path, low_memory=False)
        existing_years = set(existing["SURVEY_YEAR"].unique())
        target_years   = set(range(START_YEAR, END_YEAR + 1))
        missing_years  = sorted(target_years - existing_years)
        if not missing_years:
            print(f"  {state_abbr}: already complete, skipping.")
            return {"state": state_abbr, "rows": len(existing), "errors": []}
        print(f"  {state_abbr}: resuming — fetching {len(missing_years)} missing years.")
        years_to_fetch = missing_years
        existing_dfs   = [existing]
    else:
        years_to_fetch = list(range(START_YEAR, END_YEAR + 1))
        existing_dfs   = []

    all_dfs = list(existing_dfs)
    errors  = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {
            pool.submit(download_state_year, state_abbr, fips, yr): yr
            for yr in years_to_fetch
        }
        for future in as_completed(futures):
            result, n_rows, err = future.result()
            yr = futures[future]
            if err:
                errors.append(f"{state_abbr} {err}")
            else:
                all_dfs.append(result)

    if all_dfs:
        combined = pd.concat(all_dfs, ignore_index=True, sort=False)
        combined.to_csv(out_path, index=False)
        total = len(combined)
    else:
        total = 0

    return {"state": state_abbr, "rows": total, "errors": errors}

In [6]:
# ── Main download loop ────────────────────────────────────────────────────────

all_errors  = []
state_stats = []

for i, (abbr, fips) in enumerate(STATES, 1):
    print(f"[{i}/{len(STATES)}] Downloading {abbr}...")
    t0     = time.time()
    result = download_state(abbr, fips)
    elapsed = time.time() - t0
    state_stats.append(result)
    all_errors.extend(result["errors"])
    status = f"  → {result['rows']:,} rows in {elapsed:.0f}s"
    if result["errors"]:
        status += f"  ({len(result['errors'])} year(s) failed)"
    print(status)

# Write log
with open(LOG_FILE, "w") as f:
    if all_errors:
        f.write("\n".join(all_errors))
    else:
        f.write("No errors.")

print(f"\nDone. {len(all_errors)} total errors logged to {LOG_FILE}.")

[1/51] Downloading AL...
  → 563,003 rows in 34s
[2/51] Downloading AK...
  → 48,131 rows in 6s
[3/51] Downloading AZ...
  → 274,594 rows in 20s
[4/51] Downloading AR...
  → 455,906 rows in 29s
[5/51] Downloading CA...
  → 993,429 rows in 53s
[6/51] Downloading CO...
  → 303,483 rows in 20s
[7/51] Downloading CT...
  → 165,272 rows in 13s
[8/51] Downloading DE...
  → 36,805 rows in 5s
[9/51] Downloading DC...


C:\Users\Joshu\AppData\Local\Temp\ipykernel_5776\1772356918.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["STATE_FIPS"]  = fips        # explicit FIPS tag
C:\Users\Joshu\AppData\Local\Temp\ipykernel_5776\1772356918.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["SURVEY_YEAR"] = year
C:\Users\Joshu\AppData\Local\Temp\ipykernel_5776\1772356918.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining 

  → 10,048 rows in 3s
[10/51] Downloading FL...
  → 454,093 rows in 28s
[11/51] Downloading GA...
  → 540,100 rows in 35s
[12/51] Downloading HI...
  → 39,335 rows in 5s
[13/51] Downloading ID...
  → 151,394 rows in 13s
[14/51] Downloading IL...
  → 956,643 rows in 52s
[15/51] Downloading IN...
  → 658,889 rows in 39s
[16/51] Downloading IA...
  → 856,652 rows in 46s
[17/51] Downloading KS...
  → 887,824 rows in 49s
[18/51] Downloading KY...
  → 503,228 rows in 29s
[19/51] Downloading LA...
  → 466,148 rows in 29s
[20/51] Downloading ME...


C:\Users\Joshu\AppData\Local\Temp\ipykernel_5776\1772356918.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["SURVEY_YEAR"] = year


  → 89,858 rows in 8s
[21/51] Downloading MD...
  → 192,930 rows in 14s
[22/51] Downloading MA...
  → 184,918 rows in 14s
[23/51] Downloading MI...
  → 430,233 rows in 23s
[24/51] Downloading MN...
  → 601,855 rows in 33s
[25/51] Downloading MS...
  → 594,540 rows in 35s
[26/51] Downloading MO...
  → 862,770 rows in 48s
[27/51] Downloading MT...
  → 191,241 rows in 14s
[28/51] Downloading NE...
  → 538,740 rows in 30s
[29/51] Downloading NV...


...

  → 331,412 rows in 22s
[42/51] Downloading SD...
  → 211,000 rows in 16s
[43/51] Downloading TN...
  → 728,609 rows in 43s
[44/51] Downloading TX...
  → 1,845,885 rows in 101s
[45/51] Downloading UT...
  → 114,458 rows in 9s
[46/51] Downloading VT...
  → 98,869 rows in 9s
[47/51] Downloading VA...
  → 522,874 rows in 32s
[48/51] Downloading WA...
  → 294,628 rows in 24s
[49/51] Downloading WV...


C:\Users\Joshu\AppData\Local\Temp\ipykernel_5776\1772356918.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["SURVEY_YEAR"] = year


  → 262,817 rows in 21s
[50/51] Downloading WI...


C:\Users\Joshu\AppData\Local\Temp\ipykernel_5776\1772356918.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["STATE_FIPS"]  = fips        # explicit FIPS tag
C:\Users\Joshu\AppData\Local\Temp\ipykernel_5776\1772356918.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["SURVEY_YEAR"] = year
C:\Users\Joshu\AppData\Local\Temp\ipykernel_5776\1772356918.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining 

  → 515,747 rows in 39s
[51/51] Downloading WY...
  → 109,389 rows in 9s

Done. 0 total errors logged to download_log.txt.


In [7]:
# ── Sanity checks ────────────────────────────────────────────────────────────
stats_df = pd.DataFrame(state_stats)
print("Per-state row counts:")
print(stats_df[["state", "rows"]].sort_values("rows", ascending=False).to_string(index=False))
print(f"\nTotal rows across all states: {stats_df['rows'].sum():,}")
print(f"Expected: ~21,000,000 (620K bridges × 34 years)")

Per-state row counts:
state    rows
   TX 1845885
   OH 1028690
   CA  993429
   IL  956643
   KS  887824
   PA  872943
   MO  862770
   IA  856652
   OK  822353
   TN  728609
   NC  686493
   NY  680431
   IN  658889
   MN  601855
   MS  594540
   AL  563003
   GA  540100
   NE  538740
   VA  522874
   WI  515747
   KY  503228
   LA  466148
   AR  455906
   FL  454093
   MI  430233
   SC  331412
   CO  303483
   WA  294628
   NJ  277314
   AZ  274594
   OR  272570
   WV  262817
   SD  211000
   MD  192930
   MT  191241
   MA  184918
   CT  165272
   ND  154895
   ID  151394
   NM  141940
   UT  114458
   WY  109389
   NH  105198
   VT   98869
   ME   89858
   NV   63637
   AK   48131
   HI   39335
   DE   36805
   RI   29620
   DC   10048

Total rows across all states: 22,223,834
Expected: ~21,000,000 (620K bridges × 34 years)
